# 06 — Hyperparameter Tuning and Eight Controlled Experiments

Read the two finalists selected by Notebook 05. Evaluate each with **7 and 15 inputs** and **3 and 5 target classes**, creating eight configurations. Each configuration is tuned with **GridSearchCV** on the 240 training cases only. A separate **nested 5-fold outer / 2-fold inner CV** provides a less optimistic estimate of tuned performance (model screening across the same data still introduces selection uncertainty).

We also evaluate a majority-class baseline on the same training split for each target granularity. Never choose a winner by consulting the 60 reserved test cases.

In [1]:
from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'
MODELS = ROOT / 'models'
for p in [TABLES, FIGURES, MODELS, DATA/'processed']:
    p.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def make_preprocessor(X):
    numerical = X.select_dtypes(include='number').columns.tolist()
    categorical = [c for c in X.columns if c not in numerical]
    # Fit is performed inside each CV training fold, never before splitting.
    return ColumnTransformer([
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical),
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), numerical)
    ], remainder='drop')


In [2]:
from sklearn.model_selection import StratifiedKFold,GridSearchCV,cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier

meta=json.loads((DATA/'processed'/'experiment_config.json').read_text())
selected=json.loads((MODELS/'selected_algorithms.json').read_text())
df=pd.read_csv(DATA/'processed'/'dataset2_cleaned.csv')
split=pd.read_csv(DATA/'processed'/'split_indices.csv')
train_mask=split['split'].eq('train').to_numpy()
Y={'3class':df.loc[train_mask,meta['target']].map({int(k):v for k,v in meta['three_class_mapping'].items()}),
   '5class':df.loc[train_mask,meta['target']]}
feature_sets={'7':meta['seven_features'],'15':meta['fifteen_features']}

# Both candidates use a consistent leakage-safe preprocessing pipeline.
def build_estimator(model_name):
    available={
        'Logistic Regression':LogisticRegression(max_iter=2000,random_state=42),
        'Decision Tree':DecisionTreeClassifier(random_state=42),
        'Random Forest':RandomForestClassifier(random_state=42,n_jobs=1),
        'SVM (RBF)':SVC(),
        'KNN':KNeighborsClassifier(),
        'Gradient Boosting':GradientBoostingClassifier(random_state=42)}
    return available[model_name]

def grid_for(model_name):
    return {
      'Logistic Regression':{'model__C':[0.1,1.0,10.0], 'model__class_weight':[None,'balanced']},
      'Decision Tree':{'model__max_depth':[3,6,None], 'model__min_samples_leaf':[3,8]},
      'Random Forest':{'model__n_estimators':[80,160],'model__max_depth':[4,None],'model__min_samples_leaf':[2,5]},
      'SVM (RBF)':{'model__C':[0.1,1,10],'model__gamma':['scale',0.1]},
      'KNN':{'model__n_neighbors':[3,5,9,15],'model__weights':['uniform','distance']},
      'Gradient Boosting':{'model__n_estimators':[40,80],'model__learning_rate':[0.1],'model__max_depth':[1,2]}
    }[model_name]

inner_full=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
inner_nested=StratifiedKFold(n_splits=2,shuffle=True,random_state=73)
outer=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring={'macro_f1':'f1_macro','accuracy':'accuracy','balanced_accuracy':'balanced_accuracy'}


## Train/tune all eight combinations

The grid search chooses hyperparameters by inner-CV Macro-F1. For nested CV, the entire hyperparameter selection is repeated inside each outer training fold. This avoids reporting an already-optimized inner score as an unbiased estimate of tuning performance.

In [3]:
all_rows=[];nested_folds=[];grids=[]
for task in ['3class','5class']:
    for size,columns in feature_sets.items():
        XX=df.loc[train_mask,columns]
        yy=Y[task]
        for role in ['simple','complex']:
            model_name=selected[role]['model']
            factory=lambda: Pipeline([('preprocess',make_preprocessor(XX)),('model',build_estimator(model_name))])
            search=GridSearchCV(factory(),grid_for(model_name),cv=inner_full,scoring='f1_macro',refit=True,n_jobs=1)
            search.fit(XX,yy)
            # Outer CV reports held-out fold performance of a *tuned* pipeline.
            nested_search=GridSearchCV(factory(),grid_for(model_name),cv=inner_nested,scoring='f1_macro',refit=True,n_jobs=1)
            nested=cross_validate(nested_search,XX,yy,cv=outer,scoring=scoring,return_train_score=True,n_jobs=1)
            row={'target':task,'n_features':int(size),'model':model_name,'role':role,
                 'nested_cv_macro_f1':nested['test_macro_f1'].mean(),
                 'nested_cv_macro_f1_sd':nested['test_macro_f1'].std(),
                 'nested_cv_accuracy':nested['test_accuracy'].mean(),
                 'nested_cv_balanced_accuracy':nested['test_balanced_accuracy'].mean(),
                 'nested_cv_train_accuracy':nested['train_accuracy'].mean(),
                 'nested_cv_train_macro_f1':nested['train_macro_f1'].mean(),
                 'grid_search_cv_macro_f1':search.best_score_,
                 'best_params':json.dumps(search.best_params_,sort_keys=True),
                 'mean_outer_fit_time_seconds':nested['fit_time'].mean()}
            all_rows.append(row)
            grids.append({'target':task,'n_features':int(size),'model':model_name,
                          'grid_cv_macro_f1':float(search.best_score_),'best_params':json.dumps(search.best_params_,sort_keys=True)})
            for i in range(5):
                nested_folds.append({'target':task,'n_features':int(size),'model':model_name,
                 'fold':i+1,'val_macro_f1':nested['test_macro_f1'][i],
                 'val_accuracy':nested['test_accuracy'][i],
                 'val_balanced_accuracy':nested['test_balanced_accuracy'][i],
                 'train_accuracy':nested['train_accuracy'][i]})
            print(f"{task:6s} | {size:>2s} features | {model_name:20s} | nested macro-F1 {row['nested_cv_macro_f1']:.3f} | inner best {search.best_score_:.3f}",flush=True)


3class |  7 features | Decision Tree        | nested macro-F1 0.313 | inner best 0.326


3class |  7 features | Gradient Boosting    | nested macro-F1 0.270 | inner best 0.307


3class | 15 features | Decision Tree        | nested macro-F1 0.265 | inner best 0.337


3class | 15 features | Gradient Boosting    | nested macro-F1 0.302 | inner best 0.300


5class |  7 features | Decision Tree        | nested macro-F1 0.126 | inner best 0.173


5class |  7 features | Gradient Boosting    | nested macro-F1 0.166 | inner best 0.185


5class | 15 features | Decision Tree        | nested macro-F1 0.203 | inner best 0.217


5class | 15 features | Gradient Boosting    | nested macro-F1 0.169 | inner best 0.173


## Baseline and records for final comparison

The baseline predicts only the most frequent class. It is evaluated separately for each target definition, rather than comparing three-class scores directly to five-class scores.

In [4]:
for task in ['3class','5class']:
    XX=df.loc[train_mask,feature_sets['7']];yy=Y[task]
    baseline=Pipeline([('preprocess',make_preprocessor(XX)),('model',DummyClassifier(strategy='most_frequent'))])
    nested=cross_validate(baseline,XX,yy,cv=outer,scoring=scoring,return_train_score=True,n_jobs=1)
    row={'target':task,'n_features':0,'model':'Dummy (most frequent)','role':'baseline',
         'nested_cv_macro_f1':nested['test_macro_f1'].mean(),
         'nested_cv_macro_f1_sd':nested['test_macro_f1'].std(),
         'nested_cv_accuracy':nested['test_accuracy'].mean(),
         'nested_cv_balanced_accuracy':nested['test_balanced_accuracy'].mean(),
         'nested_cv_train_accuracy':nested['train_accuracy'].mean(),
         'nested_cv_train_macro_f1':nested['train_macro_f1'].mean(),
         'grid_search_cv_macro_f1':np.nan,'best_params':'{}',
         'mean_outer_fit_time_seconds':nested['fit_time'].mean()}
    all_rows.append(row)
    for i in range(5):
        nested_folds.append({'target':task,'n_features':0,'model':'Dummy (most frequent)',
          'fold':i+1,'val_macro_f1':nested['test_macro_f1'][i],
          'val_accuracy':nested['test_accuracy'][i],
          'val_balanced_accuracy':nested['test_balanced_accuracy'][i],
          'train_accuracy':nested['train_accuracy'][i]})

compare=pd.DataFrame(all_rows)
for task in ['3class','5class']:
    compare[compare.target.eq(task)].to_csv(TABLES/f'comparison_{task}.csv',index=False)
pd.DataFrame(nested_folds).to_csv(TABLES/'cross_validation_scores.csv',index=False)
pd.DataFrame(grids).to_csv(TABLES/'hyperparameter_results.csv',index=False)
display(compare[['target','n_features','model','nested_cv_macro_f1','nested_cv_accuracy','nested_cv_balanced_accuracy','nested_cv_train_accuracy']].round(3))


,target,n_features,model,nested_cv_macro_f1,nested_cv_accuracy,nested_cv_balanced_accuracy,nested_cv_train_accuracy
0,3class,7,Decision Tree,0.313,0.392,0.327,0.634
1,3class,7,Gradient Boosting,0.270,0.462,0.328,0.574
2,3class,15,Decision Tree,0.265,0.383,0.306,0.609
3,3class,15,Gradient Boosting,0.302,0.425,0.322,0.747
4,5class,7,Decision Tree,0.126,0.229,0.151,0.529
5,5class,7,Gradient Boosting,0.166,0.279,0.190,0.500
6,5class,15,Decision Tree,0.203,0.288,0.217,0.606
7,5class,15,Gradient Boosting,0.169,0.275,0.195,0.651
8,3class,0,Dummy (most frequent),0.223,0.504,0.333,0.504
9,5class,0,Dummy (most frequent),0.104,0.350,0.200,0.350


## Choose demo configuration for each classification task

**Predeclared practical selection rule:** Within each target, consider configurations whose nested mean Macro-F1 is at most **0.02** below the highest. Of those, prefer **fewer inputs** and then the **simpler algorithm**. This implements our preference for a practical 7-feature/simple model when predictive differences are tiny. The 0.02 threshold is a pragmatic tolerance, **not** a statistical significance test. We report every alternative instead of hiding higher-scoring models.

The held-out 20% is not used to select the demo model.

In [5]:
chosen_config={}
for task in ['3class','5class']:
    candidates=compare[(compare.target.eq(task))&(~compare.role.eq('baseline'))].copy()
    top=float(candidates['nested_cv_macro_f1'].max())
    near=candidates[candidates['nested_cv_macro_f1']>=top-.020000001].copy()
    near['role_order']=near['role'].map({'simple':0,'complex':1})
    chosen_row=near.sort_values(['n_features','role_order','nested_cv_macro_f1'],ascending=[True,True,False]).iloc[0]
    chosen_config[task]={'model':chosen_row['model'],'role':chosen_row['role'],
                         'n_features':int(chosen_row['n_features']),
                         'nested_cv_macro_f1':float(chosen_row['nested_cv_macro_f1']),
                         'best_overall_nested_cv_macro_f1':top,
                         'macro_f1_tolerance':0.02,
                         'best_params':json.loads(chosen_row['best_params'])}
    print(task,'->',chosen_config[task])
(MODELS/'demo_model_selection.json').write_text(json.dumps(chosen_config,indent=2),encoding='utf-8')
print('Saved selected configurations. Notebook 07 performs the first final holdout predictions in this workflow.')


3class -> {'model': 'Decision Tree', 'role': 'simple', 'n_features': 7, 'nested_cv_macro_f1': 0.3127686220655693, 'best_overall_nested_cv_macro_f1': 0.3127686220655693, 'macro_f1_tolerance': 0.02, 'best_params': {'model__max_depth': None, 'model__min_samples_leaf': 3}}
5class -> {'model': 'Decision Tree', 'role': 'simple', 'n_features': 15, 'nested_cv_macro_f1': 0.20329381017481704, 'best_overall_nested_cv_macro_f1': 0.20329381017481704, 'macro_f1_tolerance': 0.02, 'best_params': {'model__max_depth': None, 'model__min_samples_leaf': 3}}
Saved selected configurations. Notebook 07 performs the first final holdout predictions in this workflow.


## Limitations of the selection protocol

- Screening six algorithms on one feature/target combination is an intentional, inexpensive shortlist—not an exhaustive global optimization.
- Nested CV measures tuned pipelines more honestly than using GridSearchCV's `best_score_` alone, but selecting configurations across the same finite data still introduces uncertainty.
- The same dataset and holdout were explored in earlier project work; the final 20% scores are **not** a fully independent external evaluation.
- 3-class and 5-class targets are different tasks; compare each to its **own** baseline, not raw accuracy across tasks.